# Writing a new handler

> How to add a new data provider to the marisco pipeline.

A **handler** is a Jupyter notebook in `nbs/handlers/` that encodes one data provider's raw files into a MARIS standard NetCDF4 file. The notebook is the authoritative description of every curation decision made for that dataset. The Python module in `marisco/handlers/` is auto generated from it via nbdev.

This guide walks through the steps needed to create a handler, from prerequisites to verification. For the style and conventions used within each handler notebook, see the [handler documentation guide](../reference/handler-doc-guide.md). For the MIFA nomenclature reconciliation pattern used in several steps, see the [MIFA how-to](mifa-pattern.ipynb).

## Before you start

You will need:

- The raw source data from the provider and its format documentation.
- A Zotero record key for the dataset (8 character alphanumeric string). Create a record in the [MARIS Zotero library](https://www.zotero.org/groups/2432820/maris/library) if one does not already exist.
- The `ZOTERO_API_KEY` environment variable set.

Start with the HELCOM handler as your reference. It covers all four sample type groups and every curation rule. OSPAR, GEOTRACES, and TEPCO are additional examples that show different structural challenges.

## Step 1: Write the module header and constants

Every handler notebook starts with the `#| default_exp` directive that tells nbdev which Python module to generate. Import the standard marisco classes you will need.

In [ ]:
#| default_exp handlers.your_dataset
from fastcore.all import *
import pandas as pd
import numpy as np
from marisco.callbacks import (
    Callback, PerGroupCB, Transformer,
    EncodeTimeCB, LowerStripNameCB, SanitizeLonLatCB,
    CompareDfsAndTfmCB, RemapCB)
from marisco.metadata import GlobAttrsFeeder, BboxCB, DepthRangeCB, TimeRangeCB, ZoteroCB
from marisco.encoders import NetCDFEncoder
from marisco.match import Lut, fuzzy_merge, fix_lut, lut_from
from marisco.configs import NC_DTYPES, get_lut, lut_path, cache_path
from marisco.utils import ExtractNetcdfContents
from marisco.netcdf2csv import decode

Then define the three constants that locate the data, define the output file, and identify the dataset in Zotero.

In [ ]:
#| exports
src_dir = '...'        # path or URL to the raw data
fname_out = '...'      # default output filename
zotero_key = 'XXXXXXXX'  # 8 character Zotero record key

## Step 2: Write the data loader

The loader function reads raw provider files and returns a dictionary of DataFrames keyed by sample type group. A handler supports any subset of the four groups: SEAWATER, BIOTA, SEDIMENT, SUSPENDED_MATTER.

In [ ]:
#| export
def load_data(fname_in):
    "Load provider data; returns dict of DataFrames keyed by sample type."
    res = {}
    # Read each sample type group from the provider's files and store it
    # under the correct MARIS group key
    return res

## Step 3: Reconcile nomenclatures

Every enumerated field (nuclide name, species, body part, sediment type, unit, etc.) needs to be mapped from the provider's names to MARIS standard IDs. Follow the [MIFA pattern](mifa-pattern.ipynb) for each field. Create one fixed lookup table per field.

In [ ]:
#| exports
# Provider nuclide names that need manual correction after fuzzy matching
fixes_nuclide_names = {
    # 'provider_name': 'standard_maris_name',
}

# Lookup table factory for nuclide names
lut_nuclides = lambda df: Lut.from_df(
    fix_lut(
        fuzzy_merge(df, get_lut('nuclide'), left_on='value', right_on='nc_name'),
        fixes_nuclide_names, get_lut('nuclide'),
        left_on='value', right_on='nc_name', id_col='nuclide_id'),
    key_col='value', val_col='nuclide_id')

For small, stable enumerations with only two or three entries (for example a filtered or unfiltered flag), skip the fuzzy matching and write a plain dict directly.

## Step 4: Write callbacks

Each data quality issue in the raw input becomes one callback class. The callback docstring serves double duty: it appears as the description in the rendered documentation and is appended to the NetCDF4 audit trail at runtime.

Write the docstring as a single sentence that describes what was done and, if not obvious, why. It must stand alone as an audit trail entry.

```python
# Good — informative in both contexts
"Shift longitudes from [0, 360] convention to MARIS [-180, 180] by subtracting 180."

# Bad — too vague to serve as an audit trail entry
"Unshift longitudes."
```

Use `#| export` for callback classes and `#| exports` for lookup tables and constants passed to callbacks.

In [ ]:
#| export
class SanitizeNuclideNamesCB(PerGroupCB):
    "Lowercase and strip whitespace from provider nuclide names."
    def __init__(self, col_src, col_dst):
        store_attr()
    def each_grp(self, grp, df, tfm):
        df[self.col_dst] = df[self.col_src].str.lower().str.strip()

## Step 5: Build the Transformer pipeline

Instantiate a Transformer with the ordered list of callbacks. The order matters: sanitisation callbacks run first, then nomenclature remapping, then unit conversion and detection limit encoding, then coordinate and time validation. Each callback sees the output of the previous one.

Add `CompareDfsAndTfmCB` at the end so you can compare row counts before and after the pipeline. This surfaces unexpected row drops.

In [ ]:
dfs = load_data(src_dir)
tfm = Transformer(dfs, cbs=[
    # Sanitisation
    SanitizeNuclideNamesCB(col_src='nuclide', col_dst='NUCLIDE'),
    # Nomenclature remapping
    RemapCB(lut_nuclides, col_name='NUCLIDE'),
    # Validation
    SanitizeLonLatCB(),
    EncodeTimeCB(col_src='date'),
    # Diagnostics
    CompareDfsAndTfmCB(dfs),
])
tfm()

## Step 6: Build global attributes

The NetCDF4 file needs global attributes that describe the dataset: geographic bounding box, time range, depth range, and bibliographic metadata from Zotero.

In [ ]:
#| export
def get_attrs(tfm, zotero_key, **kw):
    "Build NetCDF global attributes for the dataset."
    return GlobAttrsFeeder(tfm.dfs, cbs=[
        BboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        ZoteroCB(zotero_key),
    ])()

## Step 7: Write the entry point

The handler exposes a single function that orchestrates the full pipeline: load, transform, build attributes, and encode.

In [ ]:
#| export
def encode(fname_out, **kwargs):
    "Encode provider data into MARIS standard NetCDF4 file."
    dfs = load_data(src_dir)
    tfm = Transformer(dfs, cbs=[...])
    tfm()
    NetCDFEncoder(
        tfm.dfs,
        dest_fname=fname_out,
        global_attrs=get_attrs(tfm, zotero_key, **kwargs)
    ).encode()

## Step 8: Verify the output

Before considering a handler complete, run through this checklist.

- All nuclide names resolve to a non-zero MARIS ID. Check with the lookup table and inspect the RemapCB output.
- `tfm.logs` contains one entry per callback. The audit trail is complete.
- Bounding box, depth range, and time range in the global attributes look geographically and temporally plausible.
- `decode(fname_out, verbose=True)` produces readable CSV files without unexpected column types.
- Row counts before and after `SanitizeLonLatCB` and `EncodeTimeCB` are consistent with the expected data volume. Use `CompareDfsAndTfmCB` to surface drops.
- Negative activity values are flagged as potential radiometric artifacts rather than silently accepted.

See [CLAUDE.md](../handlers/CLAUDE.md) for the complete technical reference on field definitions, column naming conventions, and data curation rules.

## Curation rules summary

| Rule | Detail |
|---|---|
| Units — seawater | Must be Bq m⁻³. Multiply VALUE, UNC, and any detection limit value by 1000 if the provider reports Bq L⁻¹ |
| Units — biota | Prefer wet weight (unit ID 5). Also compute PERCENTWT when both wet and dry weights are available |
| Uncertainty | 1σ absolute, same unit as VALUE. Convert from relative: UNC = VALUE × UNC_pct / 100 |
| Detection limit | Map to MARIS IDs: 1 for measured, 2 for below MDA, 3 for ND no limit, 4 for derived or aggregated |
| Coordinates | Decimal degrees. Longitude in range -180 to 180. Drop rows where latitude equals longitude equals 0 (unknown position sentinel) |
| Time | Parse to pd.Timestamp first. Encode to integer seconds since 1970-01-01 UTC via EncodeTimeCB. Drop rows with unparseable dates |
| Depth | SMP_DEPTH in metres. Use -1 as sentinel for not available |
| Unknown IDs | Set to 0 not NaN for unresolved enumerated fields. Never silently drop a row because a name could not be remapped |

## Further reading

- [MIFA pattern](mifa-pattern.ipynb) — nomenclature reconciliation step by step
- [handler documentation guide](../reference/handler-doc-guide.md) — how to document each pipeline step in the notebook